#  Learning APIs: Fetching Weather Data with Open-Meteo


## Going One Step Further

In [1]:
import requests
import pandas as pd
import plotly.express as px

print(" Welcome to the Interactive Weather Forecaster!")
print("-" * 45)

try:
    user_lat = float(input("Enter the Latitude (e.g., 59.91 for Oslo): "))
    user_lon = float(input("Enter the Longitude (e.g., 10.75 for Oslo): "))
    
    # --- STEP 1: Reverse Geocoding (Finding the City Name) ---
    print("\n🔍 Identifying your location...")
    
    # OpenStreetMap's Nominatim API for reverse geocoding
    geo_url = "https://nominatim.openstreetmap.org/reverse"
    geo_params = {
        "lat": user_lat,
        "lon": user_lon,
        "format": "json"
    }
    
    # OpenStreetMap requires a User-Agent header for their free tier so they know who is making the request
    headers = {
        "User-Agent": "StudentWeatherAppTutorial/1.0"
    }
    
    geo_response = requests.get(geo_url, params=geo_params, headers=headers)
    
    # Set a default name in case the API doesn't find a city (e.g., coordinates in the middle of the ocean)
    location_name = f"Lat: {user_lat}, Lon: {user_lon}" 
    
    if geo_response.status_code == 200:
        geo_data = geo_response.json()
        
        # Navigate the JSON dictionary to find the city, town, or village name
        if 'address' in geo_data:
            address = geo_data['address']
            # We use .get() so the code doesn't crash if 'city' isn't an exact match in the dictionary
            city = address.get('city', address.get('town', address.get('village', address.get('country', 'Unknown Area'))))
            location_name = f"{city}"
            
            # Print the full display name for the user to see in the console
            print(f"Location found: {geo_data.get('display_name', 'Unknown')}")
    else:
        print("Could not fetch the location name from the geocoding API. Using coordinates instead.")

    # --- STEP 2: Fetching the Weather Data ---
    print(f"\nFetching 7-day forecast for {location_name}...")
    
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": user_lat,
        "longitude": user_lon,
        "hourly": "temperature_2m",
        "timezone": "auto"
    }

    weather_response = requests.get(weather_url, params=weather_params)
    
    if weather_response.status_code == 200:
        weather_data = weather_response.json()
        
        # --- STEP 3: Process Data with Pandas ---
        df = pd.DataFrame(weather_data['hourly'])
        df['time'] = pd.to_datetime(df['time'])
        
        print("Data fetched successfully! Generating your interactive plot...")

        # --- STEP 4: Plotting with Plotly ---
        
        fig = px.line(
            df, 
            x="time", 
            y="temperature_2m", 
            title=f"7-Day Temperature Forecast for {location_name}", # Using our new city name here!
            labels={
                "time": "Date & Time",
                "temperature_2m": "Temperature (°C)"
            },
            template="plotly_dark"
        )
        
        fig.show()

    else:
        print(f"Oops! Weather API Error. Status Code: {weather_response.status_code}")

except ValueError:
    print("\nError: Please enter valid numbers for latitude and longitude. No letters allowed!")

 Welcome to the Interactive Weather Forecaster!
---------------------------------------------

🔍 Identifying your location...
Location found: Prinsens gate, Sjøtomta, Sentrum, Oslo, 0154, Norge

Fetching 7-day forecast for Oslo...
Data fetched successfully! Generating your interactive plot...


## Exercise

Now it's your turn! Modify the code in the cells above to do the following:

1. Change the `latitude` and `longitude` to **your home city** (You can use Google Maps or [LatLong.net](https://www.latlong.net/) to find the coordinates).
2. Add `'relative_humidity_2m'`and  `'precipitation'`to the list of `hourly` variables in the parameters.
3. Create plots that visualizes all of the above variables over time.

*Hint: Check out the [Open-Meteo Documentation](https://open-meteo.com/en/docs) to see all the different weather variables you can request!*

In [2]:
import folium

m = folium.Map(location=(45.5236, -122.6750))

In [3]:
m

In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("🌍 Welcome to the Advanced Interactive Weather Forecaster!")
print("-" * 55)

try:
    user_lat = float(input("Enter the Latitude (e.g., 59.91 for Oslo): "))
    user_lon = float(input("Enter the Longitude (e.g., 10.75 for Oslo): "))
    
    # --- STEP 1: Find the City Name (Reverse Geocoding) ---
    print("\n🔍 Identifying your location...")
    geo_url = "https://nominatim.openstreetmap.org/reverse"
    geo_params = {"lat": user_lat, "lon": user_lon, "format": "json"}
    headers = {"User-Agent": "StudentWeatherAppTutorial/1.0"}
    
    geo_response = requests.get(geo_url, params=geo_params, headers=headers)
    location_name = f"Lat: {user_lat}, Lon: {user_lon}" 
    
    if geo_response.status_code == 200:
        geo_data = geo_response.json()
        if 'address' in geo_data:
            address = geo_data['address']
            city = address.get('city', address.get('town', address.get('village', address.get('country', 'Unknown Area'))))
            location_name = f"{city}"
            print(f"📍 Location found: {geo_data.get('display_name', 'Unknown')}")

    # --- STEP 2: Fetching Multiple Weather Variables ---
    print(f"\n⛅ Fetching 7-day forecast for {location_name}...")
    
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": user_lat,
        "longitude": user_lon,
        # We now pass a list of the variables we want!
        "hourly": ["temperature_2m", "relative_humidity_2m", "precipitation"],
        "timezone": "auto"
    }

    weather_response = requests.get(weather_url, params=weather_params)
    
    if weather_response.status_code == 200:
        weather_data = weather_response.json()
        
        # --- STEP 3: Process Data with Pandas ---
        df = pd.DataFrame(weather_data['hourly'])
        df['time'] = pd.to_datetime(df['time'])
        
        print("✅ Data fetched successfully! Generating your multi-panel plot...")

        # --- STEP 4: Plotting with Plotly Subplots ---
        # Create a figure with 3 rows and 1 column, and give them shared X-axes (Time)
        fig = make_subplots(
            rows=3, cols=1, 
            shared_xaxes=False, 
            vertical_spacing=0.25,
            subplot_titles=("Temperature (°C)", "Relative Humidity (%)", "Precipitation (mm)")
        )

        # Chart 1: Temperature (Line)
        fig.add_trace(
            go.Scatter(x=df['time'], y=df['temperature_2m'], name="Temperature", line=dict(color="orange")),
            row=1, col=1
        )

        # Chart 2: Humidity (Line)
        fig.add_trace(
            go.Scatter(x=df['time'], y=df['relative_humidity_2m'], name="Humidity", line=dict(color="lightblue")),
            row=2, col=1
        )

        # Chart 3: Precipitation (Bar)
        fig.add_trace(
            go.Bar(x=df['time'], y=df['precipitation'], name="Precipitation", marker_color="blue"),
            row=3, col=1
        )

        # Update the layout to make it look clean and professional
        fig.update_layout(
            title=f"7-Day Advanced Forecast for {location_name}",
            height=800, # Make the chart taller to fit all 3 panels nicely
            showlegend=True, # We can hide the legend since the subplot titles explain what is what
            template="plotly_white"
        )
        
        fig.show()

    else:
        print(f"❌ Oops! Weather API Error. Status Code: {weather_response.status_code}")

except ValueError:
    print("\n❌ Error: Please enter valid numbers for latitude and longitude. No letters allowed!")